In [ ]:
from typing import Literal, Optional, TypedDict
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
import re
from langchain_core.tools import tool
llm = ChatOllama(model = "gpt-oss:120b-cloud")

In [ ]:
class InsuranceState(TypedDict, total=False):
    # Original customer question
    question: str

    # Supervisor routing decision
    selected_agent: Literal[
        "policy_agent",
        "claims_agent",
        "guidance_agent"
    ]

    # Extracted identifiers
    policy_number: str
    claim_number: str

    # Data returned by tools
    tool_result: dict

    # Final customer response
    answer: str

    # Error 

In [ ]:
POLICY_DATABASE = {
    "POL-1001": {
        "policy_number": "POL-1001",
        "customer_name": "Rahul Sharma",
        "policy_type": "Motor Comprehensive",
        "status": "Active",
        "coverages": [
            "Accidental damage",
            "Theft",
            "Third-party liability"
        ],
        "deductible": 5000,
        "start_date": "2026-01-01",
        "end_date": "2026-12-31"
    },

    "POL-1002": {
        "policy_number": "POL-1002",
        "customer_name": "Priya Patil",
        "policy_type": "Third-Party Motor Insurance",
        "status": "Active",
        "coverages": [
            "Third-party bodily injury",
            "Third-party property damage"
        ],
        "deductible": 0,
        "start_date": "2026-02-15",
        "end_date": "2027-02-14"
    },

    "POL-1003": {
        "policy_number": "POL-1003",
        "customer_name": "Amit Kumar",
        "policy_type": "Motor Comprehensive",
        "status": "Expired",
        "coverages": [
            "Accidental damage",
            "Theft",
            "Third-party liability"
        ],
        "deductible": 7500,
        "start_date": "2025-01-01",
        "end_date": "2025-12-31"
    }
}

In [ ]:
CLAIM_DATABASE = {
    "CLM-101": {
        "claim_number": "CLM-101",
        "policy_number": "POL-1001",
        "claim_type": "Vehicle damage",
        "incident_date": "2026-07-20",
        "status": "Submitted",
        "last_updated": "2026-07-21",
        "next_step": "Initial document verification",
        "customer_action_required": True,
        "required_action": (
            "Upload photographs of the vehicle damage."
        )
    },

    "CLM-102": {
        "claim_number": "CLM-102",
        "policy_number": "POL-1001",
        "claim_type": "Accidental damage",
        "incident_date": "2026-07-25",
        "status": "Under Assessment",
        "last_updated": "2026-08-04",
        "next_step": "Surveyor report pending",
        "customer_action_required": False,
        "required_action": None
    },

    "CLM-103": {
        "claim_number": "CLM-103",
        "policy_number": "POL-1002",
        "claim_type": "Third-party property damage",
        "incident_date": "2026-06-10",
        "status": "Approved",
        "last_updated": "2026-07-30",
        "next_step": "Payment processing",
        "customer_action_required": False,
        "required_action": None
    },

    "CLM-104": {
        "claim_number": "CLM-104",
        "policy_number": "POL-1003",
        "claim_type": "Vehicle damage",
        "incident_date": "2026-01-15",
        "status": "Rejected",
        "last_updated": "2026-02-05",
        "next_step": (
            "Contact the claims department for clarification."
        ),
        "customer_action_required": True,
        "required_action": (
            "Review the rejection details with the claims department."
        )
    }
}

In [ ]:
@tool
def lookup_policy(policy_number: str) -> dict:
    """
    Retrieve insurance policy details using a policy number.

    Use this tool for questions about policy coverage,
    status, deductible, policy type, or expiry date.
    """

    cleaned_policy_number = policy_number.strip().upper()

    policy = POLICY_DATABASE.get(cleaned_policy_number)

    if policy is None:
        return {
            "success": False,
            "error": (
                f"Policy {cleaned_policy_number} was not found."
            )
        }

    return {
        "success": True,
        "data": policy
    }

In [ ]:
@tool
def lookup_claim_status(claim_number: str) -> dict:
    """
    Retrieve the status and processing details of an existing
    insurance claim using a claim number.

    Use this tool for questions about claim status, progress,
    next steps, or customer actions.
    """

    cleaned_claim_number = claim_number.strip().upper()

    claim = CLAIM_DATABASE.get(cleaned_claim_number)

    if claim is None:
        return {
            "success": False,
            "error": (
                f"Claim {cleaned_claim_number} was not found."
            )
        }

    return {
        "success": True,
        "data": claim
    }

In [ ]:
def extract_policy_number(question: str):
    """
    Extract a policy number such as POL-1001
    from the customer question.
    """

    match = re.search(
        r"\bPOL-\d+\b",
        question.upper()
    )

    if match:
        return match.group()

    return None

In [ ]:
def extract_policy_number(question: str):
    """
    Extract a policy number such as POL-1001
    from the customer question.
    """

    match = re.search(
        r"\bPOL-\d+\b",
        question.upper()
    )

    if match:
        return match.group()

    return None

In [ ]:
SUPERVISOR_PROMPT = """
You are the routing supervisor for an Insurance Claims Assistant.

Analyze the customer's question and select exactly one agent.

Available agents:

1. policy_agent

Select policy_agent for:
- Policy coverage
- Deductible
- Policy status
- Policy expiry
- Policy type
- Questions containing a policy number such as POL-1001

2. claims_agent

Select claims_agent for:
- Existing claim status
- Claim progress
- Claim payment
- Next step for an existing claim
- Required customer action
- Questions containing a claim number such as CLM-102

3. guidance_agent

Select guidance_agent for:
- How to report an accident
- Documents required for a new claim
- Steps to start a claim
- General claim-process guidance

Return only one of these values:

policy_agent
claims_agent
guidance_agent

Do not include an explanation.
"""

In [ ]:
def supervisor_node(state: InsuranceState):
    """
    Analyze the customer question and select
    the appropriate specialist agent.
    """

    question = state.get("question", "").strip()

    if not question:
        return {
            "selected_agent": "guidance_agent",
            "error": "Customer question is missing.",
            "answer": (
                "Please enter an insurance-related question."
            )
        }

    prompt = f"""
{SUPERVISOR_PROMPT}

Customer question:
{question}
"""

    response = llm.invoke(prompt)

    selected_agent = response.content.strip().lower()

    valid_agents = [
        "policy_agent",
        "claims_agent",
        "guidance_agent"
    ]

    if selected_agent not in valid_agents:
        return {
            "selected_agent": "guidance_agent",
            "error": (
                "The supervisor returned an invalid route: "
                f"{selected_agent}"
            )
        }

    return {
        "selected_agent": selected_agent
    }

In [ ]:
def policy_agent_node(state: InsuranceState):
    """
    Handle policy-related questions using
    the Policy Lookup Tool.
    """

    question = state.get("question", "").strip()

    if not question:
        return {
            "error": "Customer question is missing.",
            "answer": (
                "Please enter a policy-related question."
            )
        }

    # Extract policy number from the question
    policy_number = extract_policy_number(question)

    if not policy_number:
        return {
            "error": "Policy number is missing.",
            "answer": (
                "Please provide your policy number. "
                "For example: POL-1001."
            )
        }

    # Call the Policy Lookup Tool
    tool_result = lookup_policy.invoke({
        "policy_number": policy_number
    })

    # Handle lookup failure
    if not tool_result["success"]:
        return {
            "policy_number": policy_number,
            "tool_result": tool_result,
            "error": tool_result["error"],
            "answer": (
                f"I could not find policy {policy_number}. "
                "Please verify the policy number and try again."
            )
        }

    policy_data = tool_result["data"]

    prompt = f"""
You are a Policy Support Agent for a motor insurance company.

Answer the customer's question using only the policy data
provided below.

Rules:
- Do not invent policy information.
- Mention whether the policy is active or expired.
- Explain the relevant coverage in simple language.
- Mention the deductible when relevant.
- Do not guarantee claim approval.
- Keep the response concise and customer-friendly.

Customer question:
{question}

Policy data:
{policy_data}
"""

    response = llm.invoke(prompt)

    return {
        "policy_number": policy_number,
        "tool_result": tool_result,
        "answer": response.content
    }

In [ ]:
def claims_agent_node(state: InsuranceState):
    """
    Handle existing claim questions using
    the Claim Status Tool.
    """

    question = state.get("question", "").strip()

    if not question:
        return {
            "error": "Customer question is missing.",
            "answer": (
                "Please enter a claim-related question."
            )
        }

    # Extract claim number from the question
    claim_number = extract_claim_number(question)

    if not claim_number:
        return {
            "error": "Claim number is missing.",
            "answer": (
                "Please provide your claim number. "
                "For example: CLM-102."
            )
        }

    # Call the Claim Status Tool
    tool_result = lookup_claim_status.invoke({
        "claim_number": claim_number
    })

    # Handle lookup failure
    if not tool_result["success"]:
        return {
            "claim_number": claim_number,
            "tool_result": tool_result,
            "error": tool_result["error"],
            "answer": (
                f"I could not find claim {claim_number}. "
                "Please verify the claim number and try again."
            )
        }

    claim_data = tool_result["data"]

    prompt = f"""
You are a Claims Support Agent for a motor insurance company.

Answer the customer's question using only the claim data
provided below.

Rules:
- Do not invent claim details.
- Clearly mention the current claim status.
- Mention the last update.
- Explain the next processing step.
- Tell the customer whether any action is required.
- Do not promise claim approval or payment.
- Keep the response concise and customer-friendly.

Customer question:
{question}

Claim data:
{claim_data}
"""

    response = llm.invoke(prompt)

    return {
        "claim_number": claim_number,
        "tool_result": tool_result,
        "answer": response.content
    }

In [ ]:
def guidance_agent_node(state: InsuranceState):
    """
    Provide general guidance about reporting an incident
    and starting an insurance claim.
    """

    question = state.get("question", "").strip()

    if not question:
        return {
            "error": "Customer question is missing.",
            "answer": (
                "Please enter a claim-guidance question."
            )
        }

    prompt = f"""
You are an Insurance Claim Guidance Agent.

Your responsibilities:
- Explain how to report an accident
- Explain how to start an insurance claim
- List commonly required documents
- Provide clear and simple next steps
- Do not approve or reject claims

Commonly required information:
- Policy number
- Incident date and time
- Incident location
- Description of the incident
- Vehicle details
- Photographs of the damage
- Police report, when applicable
- Contact details of involved parties

Customer question:
{question}
"""

    response = llm.invoke(prompt)

    return {
        "answer": response.content
    }

In [ ]:
def route_to_specialist(state: InsuranceState):
    """
    Read the Supervisor Agent's decision and select
    the next specialist node.
    """

    selected_agent = state.get(
        "selected_agent",
        "guidance_agent"
    )

    if selected_agent == "policy_agent":
        return "policy"

    if selected_agent == "claims_agent":
        return "claims"

    return "guidance"

In [ ]:
graph_builder.add_node(
    "supervisor",
    supervisor_node
)

graph_builder.add_node(
    "policy_agent",
    policy_agent_node
)

graph_builder.add_node(
    "claims_agent",
    claims_agent_node
)

graph_builder.add_node(
    "guidance_agent",
    guidance_agent_node
)

In [ ]:
# The Supervisor always runs first
graph_builder.add_edge(
    START,
    "supervisor"
)

# Route to one specialist agent
graph_builder.add_conditional_edges(
    "supervisor",
    route_to_specialist,
    {
        "policy": "policy_agent",
        "claims": "claims_agent",
        "guidance": "guidance_agent"
    }
)

# End after the selected specialist produces an answer
graph_builder.add_edge(
    "policy_agent",
    END
)

graph_builder.add_edge(
    "claims_agent",
    END
)

graph_builder.add_edge(
    "guidance_agent",
    END
)

In [ ]:
insurance_graph = graph_builder.compile()

print("Insurance Claims Assistant compiled successfully.")

In [ ]:
result = insurance_graph.invoke({
    "question": (
        "Does policy POL-1001 cover accidental damage?"
    )
})

print("Selected agent:", result.get("selected_agent"))
print("Policy number:", result.get("policy_number"))
print()
print("Answer:")
print(result.get("answer"))

In [ ]:
result = insurance_graph.invoke({
    "question": (
        "What is the deductible for policy POL-1001?"
    )
})

print("Selected agent:", result.get("selected_agent"))
print()
print("Answer:")
print(result.get("answer"))